In [2]:
import sqlite3
import warnings
import logging
from pprint import pprint
from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.google_llm import Gemini
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

from google.adk.events import Event
from google.adk.sessions import Session
from typing import Optional

class NoFunctionCallWarningFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        return "there are non-text parts in the response" not in record.getMessage()
        
logger = logging.getLogger("google_genai.types")
logger.addFilter(NoFunctionCallWarningFilter())

warnings.filterwarnings('ignore', category=UserWarning)

# Memory

In [4]:
# Sessions: Short-term memory (single conversation)
# Memory: Long-term knowledge (across multiple conversations)

### Helper Functions

In [6]:
# function that manages a complete conversation session
# - handling session creation/retrieval
# - query processing
# - response streaming
async def run_session(
    runner_instance: Runner, user_queries: list[str] | str, session_id: str = "default"
):
    """Helper function to run queries in a session and display responses."""
    print(f"\n### Session: {session_id}")

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )
    except:
        session = await session_service.get_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=session_id
        )

    # Convert single query to list
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # Process each query
    for query in user_queries:
        print(f"\nUser > {query}")
        query_content = types.Content(role="user", parts=[types.Part(text=query)])

        # Stream agent response
        async for event in runner_instance.run_async(
            user_id=USER_ID, session_id=session.id, new_message=query_content
        ):
            if event.is_final_response() and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"Model: > {text}")

In [7]:
# Retry options
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

## Memory Workflow

In [8]:
# Three-step integration process:
# 1) Initialize → Create a MemoryService and provide it to your agent via the Runner
# 2) Ingest → Transfer session data to memory using add_session_to_memory()
# 3) Retrieve → Search stored memories using search_memory()

### Initialize MemoryService

In [9]:
memory_service = (
    InMemoryMemoryService()
)  # ADK's built-in Memory Service for development and testing

In [10]:
# Define constants used throughout the notebook
APP_NAME = "agents"
USER_ID = "demo_user"

# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words.",
)


In [11]:
# Create Session Service
session_service = InMemorySessionService()  # Handles conversations

# Create runner with BOTH services
runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,  # Memory service is now available!
)

### Ingest Session Data into Memory

In [12]:
# User tells agent about their favorite color
await run_session(
    runner,
    "My favorite color is blue-green. Can you write a Haiku about it?",
    "conversation-01",  # Session ID
)


### Session: conversation-01

User > My favorite color is blue-green. Can you write a Haiku about it?
Model: > The ocean's deep hue,
A calm and serene gemstone,
Nature's gentle sigh.


In [14]:
# verify the conversation was captured in the session
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id="conversation-01"
)

# Let's see what's in the session
print("Session contains:")
for event in session.events:
    text = (
        event.content.parts[0].text[:60]
        if event.content and event.content.parts
        else "(empty)"
    )
    print(f"{event.content.role}: {text}...")

Session contains:
user: My favorite color is blue-green. Can you write a Haiku about...
model: The ocean's deep hue,
A calm and serene gemstone,
Nature's g...


In [15]:
# transfer conversation to memory
await memory_service.add_session_to_memory(session)

### Enable Memory Retrieval in Agent

In [16]:
# Create agent
user_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="MemoryDemoAgent",
    instruction="Answer user questions in simple words. Use load_memory tool if you need to recall past conversations.",
    tools=[
        load_memory
    ],  # Agent now has access to Memory and can search it whenever it decides to!
)

In [17]:
# Create a new runner with the updated agent
runner = Runner(
    agent=user_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service,
)

await run_session(runner, "What is my favorite color?", "color-test")


### Session: color-test

User > What is my favorite color?
Model: > Your favorite color is blue-green.


### Manual Memory Search

In [18]:
# search memories directly in your code
# - Debugging memory contents
# - Building analytics dashboards
# - Creating custom memory management UIs

In [19]:
# Search for color preferences
search_response = await memory_service.search_memory(
    app_name=APP_NAME, user_id=USER_ID, query="What is the user's favorite color?"
)

print("Search Results:")
print(f"Found {len(search_response.memories)} relevant memories")
print()

for memory in search_response.memories:
    if memory.content and memory.content.parts:
        text = memory.content.parts[0].text[:80]
        print(f"  [{memory.author}]: {text}...")

Search Results:
Found 2 relevant memories

  [user]: My favorite color is blue-green. Can you write a Haiku about it?...
  [MemoryDemoAgent]: The ocean's deep hue,
A calm and serene gemstone,
Nature's gentle sigh....


## Automating Memory Storage

In [21]:
# Callbacks are Python functions you define and attach to agents
# ADK automatically calls them at specific stages, acting like checkpoints during the agent's execution flow

# insert custom logic at each stage without modifying the core agent code:
# - receiving the input
# - calling the LLM
# - invoking tools
# - generating the response

In [22]:
async def auto_save_to_memory(callback_context):
    """Automatically save session to memory after each agent turn."""
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session
    )


In [23]:
# Agent with automatic memory saving
auto_memory_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="AutoMemoryAgent",
    instruction="Answer user questions.",
    tools=[preload_memory],
    after_agent_callback=auto_save_to_memory,  # Saves after each turn!
)

In [24]:
# Create a runner for the auto-save agent
# This connects our automated agent to the session and memory services
auto_runner = Runner(
    agent=auto_memory_agent,  # Use the agent with callback + preload_memory
    app_name=APP_NAME,
    session_service=session_service,  # Same services from Section 3
    memory_service=memory_service,
)

In [25]:
# Test 1: Tell the agent about a gift (first conversation)
# The callback will automatically save this to memory when the turn completes
await run_session(
    auto_runner,
    "I gifted a new toy to my nephew on his 1st birthday!",
    "auto-save-test",
)


### Session: auto-save-test

User > I gifted a new toy to my nephew on his 1st birthday!
Model: > That's wonderful! A 1st birthday is such a special milestone. I hope your nephew loves his new toy!


In [26]:
# Test 2: Ask about the gift in a NEW session (second conversation)
# The agent should retrieve the memory using preload_memory and answer correctly
await run_session(
    auto_runner,
    "What did I gift my nephew?",
    "auto-save-test-2",  # Different session ID - proves memory works across sessions!
)


### Session: auto-save-test-2

User > What did I gift my nephew?
Model: > You gifted your nephew a new toy for his 1st birthday.


## Memory Consolidation

In [29]:
# Memory Consolidation = Extracting only important facts while discarding conversational noise.
# Managed Memory Services handle consolidation automatically

In [28]:
# 1. Raw Session Events
#    ↓
# 2. LLM analyzes conversation
#    ↓
# 3. Extracts key facts
#    ↓
# 4. Stores concise memories
#    ↓
# 5. Merges with existing memories (deduplication)